In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

## Load dataset

In [2]:
x, y = fetch_california_housing(return_X_y=True)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_test, y_test, test_size=0.5, random_state=42)

In [5]:
print(f"Train set: {x_train.shape}, {y_train.shape}")
print(f"Test set: {x_test.shape}, {y_test.shape}")
print(f"Validation set: {x_val.shape}, {y_val.shape}")

Train set: (16512, 8), (16512,)
Test set: (2064, 8), (2064,)
Validation set: (2064, 8), (2064,)


## Preprocessing layer

In [16]:
import numpy as np
import torch
import torch.nn as nn


class PreprocessingLayer(nn.Module):
    def __init__(self, feature_names, excluded_features=("Latitude", "Longitude"), iqr_k=1.5):
        super().__init__()
        self.feature_names = feature_names
        self.excluded_features = set(excluded_features)

        self.iqr_k = iqr_k

        excluded_idx = [i for i, name in enumerate(feature_names) if name in self.excluded_features]
        non_excluded_idx = [i for i, name in enumerate(feature_names) if name not in self.excluded_features]

        self.register_buffer("excluded_idx", torch.tensor(excluded_idx, dtype=torch.long))
        self.register_buffer("non_excluded_idx", torch.tensor(non_excluded_idx, dtype=torch.long))
        self.register_buffer("lower_bounds", torch.empty(len(non_excluded_idx)))
        self.register_buffer("upper_bounds", torch.empty(len(non_excluded_idx)))
        self.register_buffer("mean", torch.empty(len(non_excluded_idx)))
        self.register_buffer("std", torch.empty(len(non_excluded_idx)))

    def _to_tensor(self, x):
        if isinstance(x, np.ndarray):
            return torch.tensor(x, dtype=torch.float32)
        else:
            return x.float()

    def fit(self, x_train):
        x_train = self._to_tensor(x_train)

        # Get quantiles for non-excluded features to determine outlier bounds.
        x_non_excluded = x_train[:, self.non_excluded_idx]
        q1 = torch.quantile(x_non_excluded, 0.25, dim=0)
        q3 = torch.quantile(x_non_excluded, 0.75, dim=0)
        iqr = q3 - q1

        self.lower_bounds.copy_(q1 - self.iqr_k * iqr)
        self.upper_bounds.copy_(q3 + self.iqr_k * iqr)

        inlier_mask = ((x_non_excluded >= self.lower_bounds) & (x_non_excluded <= self.upper_bounds)).all(dim=1)
        x_inliers = x_train[inlier_mask]
        x_non_excluded_inliers = x_inliers[:, self.non_excluded_idx]

        self.mean.copy_(x_non_excluded_inliers.mean(dim=0))
        std = x_non_excluded_inliers.std(dim=0)

        # Avoid division by zero
        std = torch.where(std == 0, torch.ones_like(std), std)
        self.std.copy_(std)

        return self

    def transform(self, x):
        x = self._to_tensor(x)
        x_non_excluded = x[:, self.non_excluded_idx]
        inlier_mask = ((x_non_excluded >= self.lower_bounds) & (x_non_excluded <= self.upper_bounds)).all(dim=1)

        x_used = x[inlier_mask]
        x_out = x_used.clone()

        # Standardize only non-excluded features for inliers.
        x_out[:, self.non_excluded_idx] = (x_out[:, self.non_excluded_idx] - self.mean) / self.std

        return x_out, inlier_mask

    def forward(self, x):
        return self.transform(x)


feature_names = fetch_california_housing().feature_names

preprocessor = PreprocessingLayer(feature_names=feature_names, excluded_features=("Latitude", "Longitude"), iqr_k=1.5)
preprocessor.fit(x_train)

# Get the preprocessed data
x_train_pp, train_mask = preprocessor(x_train)
x_val_pp, val_mask = preprocessor(x_val)
x_test_pp, test_mask = preprocessor(x_test)

y_train_pp = torch.tensor(y_train, dtype=torch.float32)[train_mask]
y_val_pp = torch.tensor(y_val, dtype=torch.float32)[val_mask]
y_test_pp = torch.tensor(y_test, dtype=torch.float32)[test_mask]

print(f"Train preprocessed: {x_train_pp.shape}, {y_train_pp.shape}")
print(f"Val preprocessed:   {x_val_pp.shape}, {y_val_pp.shape}")
print(f"Test preprocessed:  {x_test_pp.shape}, {y_test_pp.shape}")

Train preprocessed: torch.Size([13460, 8]), torch.Size([13460])
Val preprocessed:   torch.Size([1698, 8]), torch.Size([1698])
Test preprocessed:  torch.Size([1702, 8]), torch.Size([1702])
